# Chorus on Colab

Runs the echo-chamber simulation against a self-hosted open-weight model.
Cost is GPU time only; the model weights are free.

**Set the runtime to a GPU first:** Runtime > Change runtime type > T4 GPU.

Two things about notebooks that matter here:

- `asyncio.run()` cannot be called from inside a running event loop, and a
  notebook always has one. So the run is launched as a **subprocess**
  (`!python run_simulation.py`), not imported into a cell.
- The Colab filesystem is **ephemeral**. Cell 2 mounts Drive so a run that
  outlives a session is not lost. Skip it only for a throwaway smoke test.


## 1. Check the GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

# vLLM needs compute capability >= 7.0.
#   T4  = 7.5  OK (no bfloat16, so --dtype half is required)
#   L4  = 8.9  OK
#   A100= 8.0  OK
#   P100= 6.0  NOT SUPPORTED, vLLM will refuse to start


## 2. Persist output to Drive (recommended)

Skip this only if you do not mind losing the run when the session ends.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUNS_DIR = '/content/drive/MyDrive/chorus_runs'
!mkdir -p "$RUNS_DIR"
print('output ->', RUNS_DIR)


## 3. Install and clone

vLLM is a large install; this takes a few minutes.


In [ ]:
# Colab preinstalls a CUDA 12.x torch stack. vLLM upgrades torch to a CUDA 13
# build but leaves torchaudio behind, and transformers imports torchaudio,
# which then refuses to load against a mismatched torch:
#   RuntimeError: PyTorch and TorchAudio were compiled with different CUDA versions
#
# Remove ONLY torchaudio. Do NOT remove torchvision: vLLM imports it for its
# multimodal registry during engine startup, and without it the engine core
# dies with the unhelpful 'Engine core initialization failed'.
!pip uninstall -q -y torchaudio

!pip install -q vllm
!git clone -q https://github.com/ResearchDrafts/Chorus--Agentic-Network-Orchestration-and-Analysis.git
%cd Chorus--Agentic-Network-Orchestration-and-Analysis
!pip install -q -e .

# The red 'pip dependency resolver' wall above is Colab's preinstalled RAPIDS
# packages losing their CUDA 12 pins. None of them are used here.
print('installed. RESTART THE RUNTIME NOW, then run from the next cell.')


## 3b. Restart, then verify

The install upgrades torch, so the session must restart before vLLM can
import cleanly. **Runtime > Restart session**, then run the cell below.


In [ ]:
# Run this AFTER restarting. If it prints OK, vLLM can start.
import torch, importlib.util
print('torch      :', torch.__version__, '| CUDA', torch.version.cuda)
print('torchaudio :', 'absent (good)' if importlib.util.find_spec('torchaudio') is None else 'present')
import vllm
print('vllm       :', vllm.__version__)
print('GPU        :', torch.cuda.get_device_name(0))
print('\nOK, vLLM imports cleanly.')


## 4. Pick a model that fits the GPU

Weights at FP16, before KV cache and activations:

| Model | Weights | Fits 16 GB T4? |
|---|---|---|
| `Qwen2.5-3B-Instruct` | ~6 GB | yes, comfortably |
| `Qwen2.5-7B-Instruct` | ~15 GB | only with `--max-model-len` trimmed |
| `Qwen2.5-VL-7B-Instruct` | ~18 GB | no, needs L4/A100 or 2 GPUs |

`--max-model-len 4096` matters: prompts here run 400-800 tokens, so the
default 131K context reserves KV cache you will never use.


In [ ]:
import torch

gb = torch.cuda.get_device_properties(0).total_memory / 1e9
cc = torch.cuda.get_device_capability(0)

# 7B is ~15 GB of weights at FP16, which does not leave room for KV cache
# on a 16 GB T4 but is comfortable on a 24 GB L4 or A100.
MODEL = 'Qwen/Qwen2.5-7B-Instruct' if gb > 20 else 'Qwen/Qwen2.5-3B-Instruct'

# bfloat16 needs compute capability 8.0. Below that (T4 = 7.5) vLLM must be
# forced to fp16; at or above it, 'auto' picks the model's native dtype,
# which for Qwen is bfloat16 and numerically better than fp16.
DTYPE = 'auto' if cc >= (8, 0) else 'half'

print(f'{torch.cuda.get_device_name(0)}  {gb:.0f} GB  compute capability {cc[0]}.{cc[1]}')
print(f'  model {MODEL}')
print(f'  dtype {DTYPE}')


## 5. Start the vLLM server

Runs in the background. The wait loop below blocks until it is actually
serving, which takes a couple of minutes on first run while weights download.


In [ ]:
import subprocess, time, urllib.request, json, os, torch

# vLLM's V1 engine is the default now and assumes compute capability 8.0+
# for several of its backends. A T4 is 7.5, where V1 fails during engine
# startup with 'Engine core initialization failed'. Fall back to V0 there.
cc = torch.cuda.get_device_capability(0)
env = dict(os.environ)
if cc < (8, 0):
    env['VLLM_USE_V1'] = '0'
    print(f'compute capability {cc[0]}.{cc[1]} < 8.0, using the V0 engine')

server = subprocess.Popen(
    ['vllm', 'serve', MODEL, '--dtype', DTYPE,
     '--max-model-len', '4096', '--port', '8000'],
    env=env, stdout=open('/content/vllm.log','w'), stderr=subprocess.STDOUT)

for attempt in range(120):  # up to ~10 minutes
    try:
        with urllib.request.urlopen('http://localhost:8000/v1/models', timeout=2) as r:
            print('server up:', json.loads(r.read())['data'][0]['id'])
            break
    except Exception:
        if server.poll() is not None:
            # 'Engine core initialization failed' is a wrapper; the real
            # cause is further up, so surface that rather than the tail.
            print('server died. likely causes:')
            !grep -inE 'error|not support|capability|out of memory|bfloat|NotImplementedError' /content/vllm.log | tail -25
            print('\nfull log: /content/vllm.log')
            break
        time.sleep(5)
else:
    print('timed out; check /content/vllm.log')


## 6. Write a config

`model_backend_id` must match the served name exactly, prefixed with
`hosted_vllm/`. `api_base` is required: without it litellm routes on the
prefix alone and never reaches localhost.

Start small. M=5, K=1 is five calls and proves the whole path.


In [ ]:
import yaml, pathlib

cfg = {
    'run_id': 'colab_smoke',
    'rq_target': 'RQ1_RQ2',
    'topic': 'whether remote work should be the default for office jobs',
    'alpha': 0.5,
    'M': 5, 'N': 2, 'K': 1,
    'trial_number': 1,
    'language_condition': 'english',
    'model_backend_id': f'hosted_vllm/{MODEL}',
    'api_base': 'http://localhost:8000/v1',
    'stance_scale': [1, 2, 3, 4, 5, 6, 7],
    'stance_low_label': 'remote work should never be the default',
    'stance_high_label': 'remote work should always be the default',
    'persona_pool_id': 'test_pool',
    'meme_injection': {'enabled': False},
    'seed': 42,
    'temperature': 0.7,
    'rate_limits': {},          # empty: let vLLM batch all agents
    'checkpoint_every_n_turns': 1,
}
pathlib.Path('configs/colab_smoke.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
!python run_simulation.py configs/colab_smoke.yaml --dry-run


## 7. Run it

Subprocess, not an import: a notebook already has an event loop, so
`asyncio.run()` inside a cell would raise.


In [ ]:
RUNS = RUNS_DIR if 'RUNS_DIR' in dir() else 'runs'
!python run_simulation.py configs/colab_smoke.yaml --runs-dir "$RUNS"


## 8. Look at what came out

Three things decide whether this is worth scaling up.


In [ ]:
import pandas as pd, json, os

df = pd.read_json(f'{RUNS}/colab_smoke/interactions.jsonl', lines=True)
print(df[['speaker_agent_id','turn','stance_before','stance_after','api_call_status']])
print()
print('1. all calls succeeded :', (df.api_call_status == 'success').all())
print('2. stances moved       :', (df.stance_before != df.stance_after).any())
print('3. mean prompt tokens  :', int(df.prompt_token_count.mean()))
print()
print('--- one agent\'s reasoning ---')
print(df.reason_text.iloc[0][:400])


### Reading that

- **`failed_logged_null` rows** mean the model is not emitting the
  `STANCE: <n>` line. Some of this is normal from a 3B model. A lot of it
  means the model is too weak for this task, which matters most for
  Hinglish, where formatting is hardest.
- **Stances never moving** points at alpha or the prompt, not the model.
- **`reason_text`** should sound like the persona and address the topic.

## 9. Scaling up

Raise `M`, `K`, and `trial_number`, then sweep `language_condition` across
`english` and `hinglish` with the **same seed**, so neighbour sampling is
held constant and only language varies.

Before a real campaign, check Hinglish quality on ~100 calls. If the model
cannot produce consistent Hinglish, RQ1 measures model deficiency rather
than a language effect, and no analysis fixes that afterwards.

A run interrupted by a session timeout resumes byte-identically from its
checkpoint: re-run the same command, provided `--runs-dir` pointed at Drive.


## Shutting down


In [ ]:
server.terminate()
print('server stopped')


---
# Part 2: the real campaign

Everything above was a smoke test. This section runs the actual RQ1/RQ2
grid and stores the data.

**Design, and what is deliberately NOT here.**

- **4 topics x 2 languages**, one `ExperimentRun` per cell. Topic is an
  across-run dimension, not a within-run one, because `ExperimentRun` is
  one-run-one-condition by construction. No schema change.
- **English and Hinglish only.** That is exactly what RQ1 and RQ2 ask.
  Adding more languages would make the code-mix index undefined for most
  arms and break prompt parity, and would need its own RQ.
- **Memes are off.** RQ3 is a separate grid with `meme_injection.enabled`
  as the manipulated variable. Mixing it in here would confound RQ1.
- **1 trial to start.** Run the full grid once, look at real output, then
  raise `TRIALS` to 5 for the full design.


## 10. The grid

Each topic gets its own stance anchors. An unanchored 1-to-7 scale means
something different on immigration than on cycle lanes, so the anchors
have to be topic-specific or the scales are not comparable.

**Seeds depend on (topic, trial) but NOT on language.** That is Fix M: the
English and Hinglish arms of the same cell draw identical neighbour
samples and identical initial stances, so the only thing that differs
between them is the language directive.


In [ ]:
TOPICS = {
    'politics': dict(
        topic='whether immigration levels should be reduced',
        low='immigration levels should not be reduced at all',
        high='immigration levels should be reduced substantially'),
    'social': dict(
        topic='whether social media does more harm than good to teenagers',
        low='social media is not harmful to teenagers',
        high='social media is seriously harmful to teenagers'),
    'economic': dict(
        topic='whether the national minimum wage should be raised',
        low='the minimum wage should not be raised',
        high='the minimum wage should be raised substantially'),
    # Control: deliberately low-stakes. If stances polarize here too, the
    # driver is the echo-chamber mechanism, not the topic's divisiveness.
    'control': dict(
        topic='whether cities should expand cycle lane networks',
        low='cities should not expand cycle lanes',
        high='cities should expand cycle lanes substantially'),
}

LANGUAGES = ['english', 'hinglish']   # RQ1's manipulated variable
TRIALS    = [1]                       # raise to [1,2,3,4,5] for the full design
BASE_SEED = 2026

M, N, K, ALPHA = 100, 5, 10, 0.5      # Ohagi's baseline, unchanged

def seed_for(topic_key, trial):
    """Fix M: identical across language arms of the same cell, so
    neighbour sampling and initial stances are held constant while only
    language varies."""
    return BASE_SEED + list(TOPICS).index(topic_key) * 100 + trial

GRID = [(t, lang, tr) for t in TOPICS for lang in LANGUAGES for tr in TRIALS]
print(f'{len(GRID)} runs: {len(TOPICS)} topics x {len(LANGUAGES)} languages x {len(TRIALS)} trial(s)')


## 11. Dry run: what this will cost, before it spends anything

Equivalent to `run --dry-run` in the CLI design. Read this before running
the next cell.


In [ ]:
import json, pathlib, yaml
from sandbox.config_loader import load_run_config

CONFIG_DIR = pathlib.Path('configs/campaign'); CONFIG_DIR.mkdir(parents=True, exist_ok=True)

def write_config(topic_key, lang, trial):
    spec = TOPICS[topic_key]
    run_id = f'{topic_key}_{lang}_t{trial}'
    cfg = {
        'run_id': run_id, 'rq_target': 'RQ1_RQ2',
        'topic': spec['topic'],
        'alpha': ALPHA, 'M': M, 'N': N, 'K': K,
        'trial_number': trial,
        'language_condition': lang,
        'model_backend_id': f'hosted_vllm/{MODEL}',
        'api_base': 'http://localhost:8000/v1',
        'stance_scale': [1, 2, 3, 4, 5, 6, 7],
        'stance_low_label': spec['low'], 'stance_high_label': spec['high'],
        'persona_pool_id': 'pool_20',
        'meme_injection': {'enabled': False},   # RQ3 is a separate grid
        'seed': seed_for(topic_key, trial),
        'temperature': 0.7,
        'rate_limits': {}, 'checkpoint_every_n_turns': 1,
    }
    p = CONFIG_DIR / f'{run_id}.yaml'
    p.write_text(yaml.safe_dump(cfg, sort_keys=False))
    return p

paths = [write_config(*g) for g in GRID]
for p in paths:  load_run_config(p)          # validates every config now, not mid-campaign

calls = len(GRID) * M * K
TOK   = 950                                   # ~800 prompt + ~150 completion
for rate in (1000, 2000, 3000):
    print(f'  at {rate:>5} tok/s aggregate -> {calls*TOK/rate/3600:5.1f} GPU-hours')
print()
print(f'{len(GRID)} runs, {calls:,} calls, ~{calls*TOK/1e6:.0f}M tokens, $0.00')
print('all configs validated. MEASURE the real rate from run 1 before trusting the above.')


## 12. Run the campaign

Safe to re-run. Completed runs are skipped, and a run interrupted mid-way
resumes from its checkpoint byte-identically (Fix G plus the Phase 4
per-turn RNG derivation).

If Colab drops the session: re-run the server cell, then re-run this one.
It picks up exactly where it stopped.


In [ ]:
import time, urllib.request
from sandbox.config_loader import load_run_config
from sandbox.cost_tracker import CostTracker
from sandbox.model_gateway import ModelGateway
from sandbox.rate_limiter import RateLimiter
from sandbox.simulation_orchestrator import build_orchestrator

RUNS = pathlib.Path(RUNS_DIR if 'RUNS_DIR' in dir() else 'runs')

def already_done(run_id):
    """status is written only when a run reaches the end (Phase 4 2c)."""
    p = RUNS / run_id / 'run_config.json'
    return p.exists() and json.loads(p.read_text()).get('status') == 'completed'

def server_alive():
    try:
        urllib.request.urlopen('http://localhost:8000/v1/models', timeout=3); return True
    except Exception:
        return False

assert server_alive(), 'vLLM is not responding. Re-run the server cell first.'

for i, path in enumerate(paths, 1):
    run = load_run_config(path)
    if already_done(run.run_id):
        print(f'[{i}/{len(paths)}] {run.run_id}: already complete, skipping'); continue
    if not server_alive():
        print('server went away. re-run the server cell, then re-run this cell.'); break

    print(f'[{i}/{len(paths)}] {run.run_id}  ({run.language_condition}, seed {run.seed})', flush=True)
    t0 = time.time()
    tracker = CostTracker(run)
    gateway = ModelGateway(run.model_backend_id, RateLimiter(run.rate_limits),
                           tracker, api_base=run.api_base)
    await build_orchestrator(run, gateway, cost_tracker=tracker, runs_dir=RUNS).run()
    mins = (time.time() - t0) / 60
    print(f'      done in {mins:.1f} min  ({M*K/(mins*60):.1f} calls/s)', flush=True)

print('\ncampaign finished (or paused). re-run this cell to continue.')


### Note on `await` here

This cell uses top-level `await` rather than `run_simulation.py`, because a
notebook already has an event loop and `asyncio.run()` would raise inside
it. Colab supports top-level `await`, so calling the orchestrator directly
is the correct form here. The script is for terminals.


## 13. Combine every run into one DataFrame

`interactions.jsonl` does not carry `topic` or `language_condition`, since
those live once per run in `run_config.json`. This joins them on so you can
group by either.

This is a stand-in for Phase 5's `load_run_dataframe`, which is not built
yet. Deliberately kept to the shape that function will have, so it lifts
into `sandbox/analysis/loader.py` unchanged rather than becoming a second
implementation.


In [ ]:
import pandas as pd

def load_run_dataframe(run_dir):
    """One run's interactions, with its run-level config joined on."""
    cfg = json.loads((run_dir / 'run_config.json').read_text())
    df = pd.read_json(run_dir / 'interactions.jsonl', lines=True)
    for col in ('topic', 'language_condition', 'trial_number', 'seed', 'run_id', 'alpha'):
        df[col] = cfg[col]
    df['topic_key'] = run_dir.name.split('_')[0]
    return df

completed = [d for d in sorted(RUNS.iterdir())
             if (d / 'run_config.json').exists() and already_done(d.name)]
df = pd.concat([load_run_dataframe(d) for d in completed], ignore_index=True)

out = RUNS / 'combined_interactions.parquet'
df.to_parquet(out)
print(f'{len(completed)} runs -> {len(df):,} interactions -> {out}')
df.head(3)


## 14. Did stance actually change?

The first question worth asking of real data, before any modelling.


In [ ]:
df['shift'] = (df.stance_after - df.stance_before).abs()
valid = df[df.api_call_status != 'failed_logged_null']

print('mean absolute stance shift per agent-turn\n')
print(valid.pivot_table(index='topic_key', columns='language_condition',
                        values='shift', aggfunc='mean').round(3))
print('\nfinal-turn stance spread (higher = more polarized)\n')
last = valid[valid.turn == valid.turn.max()]
print(last.pivot_table(index='topic_key', columns='language_condition',
                       values='stance_after', aggfunc='std').round(3))
print('\nparse-failure rate by language (an uneven rate is itself a confound)\n')
print(df.groupby('language_condition')
        .apply(lambda g: (g.api_call_status == 'failed_logged_null').mean(), include_groups=False)
        .round(4))
print('\nmean prompt tokens by language (Fix A truncation-symmetry check)\n')
print(valid.groupby('language_condition').prompt_token_count.mean().round(1))


## 15. Back up off Colab

Drive already persists this, so the zip is a second copy you can pull down
and keep with the paper. Cheap insurance against a Drive sync problem.


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive('/content/chorus_runs', 'zip', RUNS)
print(f'{archive}  ({pathlib.Path(archive).stat().st_size/1e6:.1f} MB)')
files.download(archive)


## Scaling up from here

1. Check the tables in cell 14. Stances should move, and the parse-failure
   rate should be similar across languages. A much higher Hinglish failure
   rate means the model struggles with the format in that condition, which
   is a finding about the model, and it biases RQ1 if left unaddressed.
2. Read a few `reason_text` values from each language arm by hand. This is
   the A.10 Hinglish gate: is the Hinglish genuinely code-mixed, or English
   with a few Hindi words? If the latter, RQ1 is measuring nothing.
3. Set `TRIALS = [1,2,3,4,5]` and re-run cell 12. Completed runs are
   skipped, so only the 32 new ones execute.
4. RQ3 is a separate grid: same topics, `language_condition` fixed, and
   `meme_injection.enabled` toggled across runs.
